In [2]:
# imports
import numpy as np
import numpy.linalg as LA
import pandas as pd

from base.qda import QDA, TensorizedQDA, FasterQDA, EfficientQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.datasets import get_wine_dataset, label_encode
from utils.bench import Benchmark

In [3]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)

In [4]:
benchmark = Benchmark(X_full, y_full_encoded)

modelos = [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
           QDA_Chol1, QDA_Chol2, QDA_Chol3]

for modelo in modelos:
    benchmark.bench(modelo)

Benching params:
Total runs: 1200
Warmup runs: 100
Peak Memory usage runs: 100
Running time runs: 1000
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

...

QDA_Chol1 (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/1000 [00:00<?, ?it/s]

In [5]:
benchmark.summary(baseline="QDA")

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.145604,0.059463,1.681750,0.115360,0.984593,0.018173,0.000834,0.007697,0.000343,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.149271,0.073857,0.790584,0.103933,0.984593,0.018082,0.000800,0.011986,0.000212,0.975437,2.127226,1.005063,0.642187
FasterQDA,0.143334,0.552457,0.036625,0.036017,0.984593,0.018082,0.000821,0.109116,0.000250,1.015844,45.917995,1.005063,0.070541
EfficientQDA,0.142917,0.050557,0.035417,0.021906,0.984593,0.018082,0.000789,0.060089,0.000256,1.018805,47.484370,1.005063,0.128095
QDA_Chol1,0.174354,0.135067,0.965625,0.467398,0.984593,0.018272,0.000841,0.007767,0.000249,0.835108,1.741618,0.994572,0.991036
QDA_Chol2,0.160104,0.080047,2.321250,1.157806,0.984593,0.018105,0.000823,0.007968,0.000208,0.909437,0.724502,1.003793,0.966008
QDA_Chol3,0.145458,0.060169,0.963166,1.028385,0.984593,0.018105,0.000814,0.007622,0.000223,1.001007,1.746064,1.003793,1.009885


## 8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

Dada la matriz simétrica y definida positiva $A \in \mathbb{R}^{p \times p}$ tal que 
$$
A = LL^T
$$ 

siendo $L$ una matriz triangular inferior.

El determinante de $L$ es el producto de su diagonal, y en la factorización de Cholesky todos los elementos de la diagonal son positivos (ya que $A$ es definida positiva). Por lo tanto, $L$ tiene determinante distinto de 0 y es invertible.

Por la propiedad de la inversa de un producto de matrices:

$$
A^{-1} = (LL^T)^{-1} = (L^T)^{-1} L^{-1}
$$

y como $(L^T)^{-1} = (L^{-1})^T$, se obtiene:

$$
A^{-1} = (L^{-1})^T L^{-1}
$$


Teniendo QDA definida como:
$$
\delta_k(x) = -\frac{1}{2} \ln |\Sigma_k| - \frac{1}{2} (x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k) + \ln \pi_k
$$

se necesita invertir $\Sigma_k$ para cada clase. Invertir matrices con un método genérico es computacionalmente costoso, además de numéricamente inestable. Por lo visto anterior, Cholesky permite descomponer $\Sigma_k = L_k L_k^T$, con $L_k$ triangular inferior, quedando etonces:

$$
\delta_k(x) = -\frac{1}{2} \ln |L_k L_k^T| - \frac{1}{2} (x - \mu_k)^T (L_k^{-1})^T L_k^{-1} (x - \mu_k) + \ln \pi_k
$$

Dado que $L_k$ es triangular, su determinante es el producto de su diagonal:

$$
|\Sigma_k| = |L_k L_k^T| = |L_k|^2 = \left( \prod_{i=1}^{p} (L_k)_{ii} \right)^2
$$

por lo que el primer término se reduce a:

$$
-\frac{1}{2} \ln |\Sigma_k| = -\ln \prod_{i=1}^{p} (L_k)_{ii}
$$

Por la propiedad de la traspuesta de un producto, se tiene que 
$$
(x - \mu_k)^T (L_k^{-1})^T = \left( L_k^{-1} (x - \mu_k) \right)^T
$$ 

Definiendo un vector $z_k$ como $z_k = L_k^{-1} (x - \mu_k)$, el término cuadrático queda como la norma al cuadrado de ese vector:

$$
\frac{1}{2} (x - \mu_k)^T (L_k^{-1})^T L_k^{-1} (x - \mu_k) = \frac{1}{2} z_k^T z_k = \frac{1}{2} \|z_k\|^2
$$

$z_k$ puede obtenerse resolviendo el sistema de ecuaciones lineales:

$$
L_k z_k = x - \mu_k
$$

Dado que $L_k$ es triangular inferior, este sistema se resuelve por sustitución hacia adelante: resuelvo la primer fila con el único término distinto a 0 siendo la primer componente de $z_k$, luego reemplazo en la segunda fila para que quede una sola incógnita y resuelvo para la segunda componente, y así sucesivamente. No hay necesidad de calcular ninguna matriz inversa.

La factorización de Cholesky también tiene costo $O(p^3)$, pero requiere aproximadamente un tercio de las operaciones de una inversión normal y es numéricamente más estable. Además, una vez obtenida $L_k$, el determinante y la forma cuadrática se calculan de forma mucho más barata.



## 9. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
Ambas `QDA` y `QDA_Chol1` heredan de `BaseBayesianClassifier` y comparten un flujo general. Las diferencias están en `_fit_params` (que define qué es lo que guarda al entrenar el modelo) y `_predict_log_conditional` (que define cómo se calcula el puntaje de cada clase).

**Fit**

Para cada clase k, ambos modelos seleccionan las observaciones que pertencen a esa clase y calculan la matriz de covarianza $\Sigma_k$ y su media $\mu_k$.

`QDA` calcula y almacena la inversa de cada matriz de covarianzas $\Sigma_k$.

Mientras tanto, `QDA_Chol1` utiliza `cholesky` para expresar a la matriz $\Sigma$ como el producto de una matriz triangular inferior $L_k$ y su traspuesta. Luego calcula la inversa de esta matriz $L_k$ (curiosamente, de forma genérica en vez de aprovechar su forma) para cada clase y las almacena.
En ambos casos se realiza la inversión de una matriz, la diferencia es qué matriz se invierte.

**Predict**

Ambos modelos heredan un flujo base: para cada observación $x$ se recorren todas las clases y se calcula el puntaje de cada una con `_predict_log_conditional`. Se suma el logaritmo de su probabilidad y se predice la clase como aquella que tenga mayor puntaje con `argmax`.

Ambos calculan `unbiased_x = x - self.means[class_idx]` ($x - \mu_k$).

Para calcular el determinante, `QDA` lo hace de forma genérica.

`QDA_Chol1` aprovecha la forma de la matriz  $L_k$ para calcular determinantes usando solo el producto de la diagonal con `np.log(L_inv.diagonal().prod())`. Como la diagonal $L_k^{-1}$ es la diagonal de $L_k$ invertida, este calor da lo mismo que el término $-\frac{1}{2} \ln |\Sigma_k|$ de `QDA` (por eso aparece con signo positivo).

Para el término cuadrático, `QDA_Chol1` multiplica $L_k^{-1}$ por $x - \mu_k$ con `y = L_inv @ unbiased_x`, que es el vector $z_k$ del punto (8). Después calcula su norma al cuadrado con `(y**2).sum()`. En cambio, `QDA` multiplica por la matriz $\Sigma_k$ inversa entera.

Ambos modelos llegan al mismo puntaje para cada clase y realizan las mismas predicciones.

## 10. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
Las 3 versiones de QDA utilizan `cholesky`, pero lo aprovechan de formas distintas.

`QDA_Chol1` calcula la inversa de la matriz $L_k$ de forma genérica, sin aprovechar el hecho de que $L_k$ es una matriz triangular inferior. Luego calcula `y = L_inv @ unbiased_x` en el paso predictivo para utilizar en el término cuadrático.

`QDA_Chol2` evita el paso de calcular la inversa de $L_k$ en `_fit_params` y almacena a $L_k$ directamente. Al pasar al paso predictivo, utiliza `solve_triangular` de la librería `scipy` para resolver el sistema de ecuaciones y obtener `y`. Además, el término del logaritmo del determinante lleva signo negativo dado que lo calcula con $L_k$ en vez de $L_k^{-1}$ (y $\ln |L_k^{-1}| = -\ln |L_k|$).

`QDA_Chol3` es más similar a `QDA_Chol1`, con la diferencia de que sí aprovecha la forma de la matriz triangular inferior. En este caso, el modelo calcula y almacena la inversa de $L_k$ pero lo hace con `dtrtri` de la librería `scipy`. `dtrtri` es más barata que `inv`.

`QDA_Chol2` tiene un fit más "liviano" pero tiene un costo extra al realizar la predicción (resolver un sistema de ecuaciones en vez de multiplicar en cada una de sus llamadas).

`QDA_Chol1` y `QDA_Chol3` predicen de la misma forma, la diferencia radica en que `QDA_Chol3` calcula la inversa de $L_k$ de la forma más barata.


## 11. Comparar las variantes de QDA
Todas las variantes devuelven una misma accuracy (0.984593) dado que predicen lo mismo a pesar de hacerlo de formas distintas.
Con el dataset de wine, la diferencia de entrenamiento entre QDA y los modelos con Cholesky no es tan notoria. `QDA_Chol3` parece ser ligeramente más rápido al entrenar que los otros modelos con Cholesky, pero es una diferencia pequeña.
Esto se debe a que esto se debe a que el dataset de wine tiene 13 features y con una matriz 13x13 no llega a notarse el beneficio de Cholesky ya que invertirla es rápido. Para un dataset con p más grande el resultado sería más claro.
En el caso de la predicción sí es más clara la diferencia en comparación Cholesky. `QDA_Chol1` y `QDA_Chol3` con casi el doble de rápidos que `QDA` al predecir. `QDA` calcula su determinante de forma genérica, mientras que `QDA_Chol1` y `QDA_Chol3` solo tienen que sacar el producto de la diagonal. En el término cuadrático, `QDA` tiene que realizar dos productos de matrices mientras que `QDA_Chol1` y `QDA_Chol3` calculan un solo producto de matrices (`y = L_inv @ unbiased_x`) y luego la norma al cuadrado de `y`.
Por otro lado, `QDA_Chol2` es incluso más lento que `QDA` al predecir. Esto se debe a que `solve_triangular` tiene un costo fijo por cada llamado y se tienen que llamar nxk veces.
De todas formas, ninguo de los modelos con Cholesky rindieron mejor que `EfficientQDA`, que cuenta con la ventaja de eliminar los cíclos `for`.
De tener que seleccionar un modelo de Cholesky, `QDA_Chol3` es el más equilibrado al entrenar y predecir mientras que `QDA_Chol2` no presenta suficientes ventajas al entrenar como para amortiguar lo que tarda al predecir.
Lo interesante será ver qué sucede al combinar las ventajas de utilizar Cholesky con la vectorización de `EfficientQDA` en la sección (4) del trabajo.

In [1]:
# Versiones utilizadas
import sys, numpy, scipy
print(f"Python {sys.version.split()[0]}")
print(f"NumPy  {numpy.__version__}")
print(f"SciPy  {scipy.__version__}")

Python 3.12.14
NumPy  2.3.1
SciPy  1.16.0
